# Bayesian Socio-Geodemographic SSE Regression

It uses the reusable library modules to prepare model frames, formulas, output paths, Bambi fits, posterior summaries, diagnostics, and saved result tables.

The notebook is arranged by model family and domain:

- Logistic regression: `candidate` as the outcome.
- Linear regression: `burst_score` and `burden_score` as outcomes.
- Mixing models: node-level entropy/context predictors.
- Composition models: sequence-level socio-geodemographic predictors.


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import sys

os.environ.setdefault("JAX_PLATFORMS", "cpu")

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "utils").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "utils").exists():
    raise RuntimeError("Run this notebook from inside the scotland repository.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sse_detection.lib import (  # noqa: E402
    BayesianFitConfig,
    SampleSpec,
    fit_prepared_model,
    load_sse_outputs,
    prepare_regression_data,
    prepare_regression_run,
    save_prepared_model_result,
)

SSE_OUTPUT_DIR = PROJECT_ROOT / "sse_detection" / "results" / "sse_outputs"
RESULT_DIR = (
    PROJECT_ROOT / "sse_detection" / "results" / "bayesian_socio_geo_demo_regression"
)
LOGISTIC_RESULT_DIR = RESULT_DIR / "logistic"
LINEAR_RESULT_DIR = RESULT_DIR / "linear"

## Load and Align Data

Nodes at least as large as the smallest high-priority burst/burden candidate. Sequence-level composition rows inherit the candidate label and score outcomes from their cluster.


In [ ]:
sse_outputs = load_sse_outputs(SSE_OUTPUT_DIR)
regression_data = prepare_regression_data(sse_outputs)

print(f"Minimum candidate cluster size: {regression_data.min_candidate_size}")
print(regression_data.eligibility_summary)

## Sampling and Fit Configuration

The notebook uses a small fraction of data for every complete-case frame. Logistic samples preserve the observed candidate fraction by default; composition samples also seed categorical levels so treatment-coded terms stay valid.

Increase `rows`, `draws`, or `tune` when moving from smoke-test to final analysis.

In [ ]:
RANDOM_SEED = 123
DISPLAY_TABLES = True
SAVE_INFERENCE_DATA = False

FIT_CONFIG = BayesianFitConfig(
    draws=500,
    tune=500,
    chains=4,
    cores=4,
    target_accept=0.99,
    random_seed=RANDOM_SEED,
    inference_method="pymc",
)

mixing_rate = float(regression_data.eligible_nodes["candidate"].mean())
composition_rate = float(regression_data.eligible_sequence_data["candidate"].mean())

MIXING_SAMPLE = SampleSpec(
    rows=10_000,
    positive_fraction=mixing_rate,
    random_state=RANDOM_SEED,
)
COMPOSITION_SAMPLE = SampleSpec(
    rows=10_000,
    positive_fraction=composition_rate,
    random_state=RANDOM_SEED,
)

print(f"Mixing candidate rate: {mixing_rate:.3%}")
print(f"Composition candidate rate: {composition_rate:.3%}")

## Build Model Frames and Formula Grids

This creates complete-case frames, sampled fit frames, Bambi formulas, and organized output directories for all available configurations.


In [ ]:
logistic_run = prepare_regression_run(
    regression_data,
    family="logistic",
    result_dir=LOGISTIC_RESULT_DIR,
    mixing_sample=MIXING_SAMPLE,
    composition_sample=COMPOSITION_SAMPLE,
    write_tables=True,
)

linear_run = prepare_regression_run(
    regression_data,
    family="linear",
    result_dir=LINEAR_RESULT_DIR,
    mixing_sample=MIXING_SAMPLE,
    composition_sample=COMPOSITION_SAMPLE,
    write_tables=True,
)

print("Logistic model grid")
print(logistic_run.model_grid)
print("Logistic fit-frame summary")
print(logistic_run.fit_frame_summary)

print("Linear model grid")
print(linear_run.model_grid)
print("Linear fit-frame summary")
print(linear_run.fit_frame_summary)

## Shared Fitting Helper

Each model cell below calls this helper. It fits one prepared frame, prints diagnostics/posterior summaries, and writes `summary.csv`, `diagnostics.csv`, and `metadata.csv` under the frame's configured output directory.


In [ ]:
FIT_RESULTS = {}
MANIFEST_ROWS = []


def fit_and_save_frame(prepared, *, domain: str, outcome: str, model_set: str):
    """Fit one prepared model frame and save the standard output files."""
    frame = prepared.select(domain=domain, outcome=outcome, model_set=model_set)
    key = f"{frame.family}:{domain}:{outcome}:{model_set}"
    print("=" * 100)
    print(key)
    print(frame.formula)
    print(
        f"Fit rows: {len(frame.fit_df):,} / complete-case rows: {len(frame.full_df):,}"
    )
    print(f"Output dir: {frame.output_dir.relative_to(PROJECT_ROOT)}")

    result = fit_prepared_model(
        frame,
        config=FIT_CONFIG,
        display_tables=DISPLAY_TABLES,
        print_diagnostics=True,
    )
    manifest_row = save_prepared_model_result(
        result,
        frame,
        save_idata=SAVE_INFERENCE_DATA,
    )
    FIT_RESULTS[key] = result
    MANIFEST_ROWS.append(manifest_row)
    return result

# Logistic Candidate Models

Outcome: `candidate`.

The primary models include the focal composition or mixing predictors. Expanded models add epidemic/surveillance context adjusters.

## Logistic Mixing: Null Primary

Node-level candidate association with null-standardised mixing features.

In [ ]:
logistic_mixing_null_primary = fit_and_save_frame(
    logistic_run,
    domain="mixing",
    outcome="candidate",
    model_set="null_primary",
)

## Logistic Mixing: Null Expanded

Node-level candidate association with null-standardised mixing features plus context adjusters.


In [ ]:
logistic_mixing_null_expanded = fit_and_save_frame(
    logistic_run,
    domain="mixing",
    outcome="candidate",
    model_set="null_expanded",
)

## Logistic Mixing: Observed Primary

Node-level candidate association with observed entropy scales.


In [ ]:
logistic_mixing_observed_primary = fit_and_save_frame(
    logistic_run,
    domain="mixing",
    outcome="candidate",
    model_set="observed_primary",
)

## Logistic Mixing: Observed Expanded

Node-level candidate association with observed entropy scales plus context adjusters.


In [ ]:
logistic_mixing_observed_expanded = fit_and_save_frame(
    logistic_run,
    domain="mixing",
    outcome="candidate",
    model_set="observed_expanded",
)

## Logistic Composition: Primary

Sequence-level candidate association with sex, age band, SIMD quintile, urban/rural class, and health board.


In [16]:
logistic_composition_primary = fit_and_save_frame(
    logistic_run,
    domain="composition",
    outcome="candidate",
    model_set="primary",
)

ValueError: Not enough samples to build a trace.

## Logistic Composition: Expanded

Sequence-level candidate association with composition predictors plus context adjusters.


In [ ]:
logistic_composition_expanded = fit_and_save_frame(
    logistic_run,
    domain="composition",
    outcome="candidate",
    model_set="expanded",
)

# Linear Score Models

Outcomes: `burst_score` and `burden_score`.

These use the same mixing/composition predictor sets as the logistic candidate models, but report coefficient direction probabilities rather than odds ratios.


## Linear Mixing: Burst Score / Null Primary


In [ ]:
linear_mixing_burst_null_primary = fit_and_save_frame(
    linear_run,
    domain="mixing",
    outcome="burst_score",
    model_set="null_primary",
)

## Linear Mixing: Burst Score / Null Expanded


In [ ]:
linear_mixing_burst_null_expanded = fit_and_save_frame(
    linear_run,
    domain="mixing",
    outcome="burst_score",
    model_set="null_expanded",
)

## Linear Mixing: Burst Score / Observed Primary


In [ ]:
linear_mixing_burst_observed_primary = fit_and_save_frame(
    linear_run,
    domain="mixing",
    outcome="burst_score",
    model_set="observed_primary",
)

## Linear Mixing: Burst Score / Observed Expanded


In [ ]:
linear_mixing_burst_observed_expanded = fit_and_save_frame(
    linear_run,
    domain="mixing",
    outcome="burst_score",
    model_set="observed_expanded",
)

## Linear Mixing: Burden Score / Null Primary


In [ ]:
linear_mixing_burden_null_primary = fit_and_save_frame(
    linear_run,
    domain="mixing",
    outcome="burden_score",
    model_set="null_primary",
)

## Linear Mixing: Burden Score / Null Expanded


In [ ]:
linear_mixing_burden_null_expanded = fit_and_save_frame(
    linear_run,
    domain="mixing",
    outcome="burden_score",
    model_set="null_expanded",
)

## Linear Mixing: Burden Score / Observed Primary


In [ ]:
linear_mixing_burden_observed_primary = fit_and_save_frame(
    linear_run,
    domain="mixing",
    outcome="burden_score",
    model_set="observed_primary",
)

## Linear Mixing: Burden Score / Observed Expanded


In [ ]:
linear_mixing_burden_observed_expanded = fit_and_save_frame(
    linear_run,
    domain="mixing",
    outcome="burden_score",
    model_set="observed_expanded",
)

## Linear Composition: Burst Score / Primary


In [ ]:
linear_composition_burst_primary = fit_and_save_frame(
    linear_run,
    domain="composition",
    outcome="burst_score",
    model_set="primary",
)

## Linear Composition: Burst Score / Expanded


In [ ]:
linear_composition_burst_expanded = fit_and_save_frame(
    linear_run,
    domain="composition",
    outcome="burst_score",
    model_set="expanded",
)

## Linear Composition: Burden Score / Primary


In [ ]:
linear_composition_burden_primary = fit_and_save_frame(
    linear_run,
    domain="composition",
    outcome="burden_score",
    model_set="primary",
)

## Linear Composition: Burden Score / Expanded


In [ ]:
linear_composition_burden_expanded = fit_and_save_frame(
    linear_run,
    domain="composition",
    outcome="burden_score",
    model_set="expanded",
)

# Saved Model Manifest

This collects the rows written by each fitting cell. Re-run this after fitting any subset of models.


In [ ]:
saved_model_manifest = pd.DataFrame(MANIFEST_ROWS)
if not saved_model_manifest.empty:
    combined_manifest_path = COMBINED_RESULT_DIR / "saved_model_manifest.csv"
    saved_model_manifest.to_csv(combined_manifest_path, index=False)
    print(
        f"Saved combined manifest to: {combined_manifest_path.relative_to(PROJECT_ROOT)}"
    )
display(saved_model_manifest)